In [1]:
print("Hello, I am ready to build a GPT!")

Hello, I am ready to build a GPT!


In [1]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.11.0+cu128
GPU available: True


In [2]:
# Download the Shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# Read it and print basic info
with open('input.txt', 'r') as f:
    text = f.read()

print("Total number of characters:", len(text))
print("\nFirst 500 characters of the dataset:")
print(text[:500])

--2026-06-05 06:15:44--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-06-05 06:15:44 (152 MB/s) - ‘input.txt’ saved [1115394/1115394]

Total number of characters: 1115394

First 500 characters of the dataset:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have co

In [3]:
# Get all unique characters in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("All unique characters:")
print(''.join(chars))
print("\nTotal unique characters (vocab size):", vocab_size)

# Create two lookup tables:
# stoi = character to number (string to integer)
# itos = number to character (integer to string)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# encode = converts text to list of numbers
# decode = converts list of numbers back to text
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Test it!
print("\nEncoding 'Hello':", encode('Hello'))
print("Decoding back:", decode(encode('Hello')))

All unique characters:

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Total unique characters (vocab size): 65

Encoding 'Hello': [20, 43, 50, 50, 53]
Decoding back: Hello


In [4]:
import torch

# Convert all the text into numbers using our encoder
data = torch.tensor(encode(text), dtype=torch.long)

print("Shape of data:", data.shape)
print("First 100 numbers:", data[:100])

# Split into 90% train and 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("\nTraining data size:", len(train_data))
print("Validation data size:", len(val_data))

Shape of data: torch.Size([1115394])
First 100 numbers: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

Training data size: 1003854
Validation data size: 111540


In [5]:
# Hyperparameters (settings for our model)
block_size = 8   # how many characters the model sees at once
batch_size = 4   # how many chunks we process at the same time

def get_batch(split):
    # pick train or validation data
    data = train_data if split == 'train' else val_data

    # pick random starting positions
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # x = input characters
    x = torch.stack([data[i:i+block_size] for i in ix])

    # y = target characters (shifted by 1)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

# Test it!
xb, yb = get_batch('train')
print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("\nInput batch:")
print(xb)
print("\nTarget batch:")
print(yb)

Input shape: torch.Size([4, 8])
Target shape: torch.Size([4, 8])

Input batch:
tensor([[ 5,  0, 13, 52, 42,  1, 57, 47],
        [56,  1, 58, 46, 56, 47, 44, 58],
        [43, 57, 58,  1, 58, 53,  1, 42],
        [46, 43,  1, 51, 39, 52, 59, 39]])

Target batch:
tensor([[ 0, 13, 52, 42,  1, 57, 47, 52],
        [ 1, 58, 46, 56, 47, 44, 58, 57],
        [57, 58,  1, 58, 53,  1, 42, 53],
        [43,  1, 51, 39, 52, 59, 39, 50]])


In [6]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each character looks up the next character probabilities
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # get predictions
        logits = self.token_embedding_table(idx)  # shape: (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Create the model
model = BigramLanguageModel(vocab_size)

# Test it with our batch
logits, loss = model(xb, yb)
print("Output shape:", logits.shape)
print("Initial loss:", loss.item())
print("\nExpected loss should be around 4.17")

Output shape: torch.Size([32, 65])
Initial loss: 4.523599147796631

Expected loss should be around 4.17


In [7]:
# Generate some text before training
# Start with character 0 (new line character)
context = torch.zeros((1, 1), dtype=torch.long)

# Generate 200 characters
generated = model.generate(context, max_new_tokens=200)

# Convert numbers back to text
print("Generated text BEFORE training:")
print("="*40)
print(decode(generated[0].tolist()))
print("="*40)
print("\nAs expected - complete garbage! This is BEFORE training.")

Generated text BEFORE training:

uXODZgloFojKcFL,.OfVfSyUj aIno:M'Fp,wqV.SBE'Wq qVlfTamgrkPi
GaCSRo!L:L tauwDKg&FUb:QEJvsM-mH$MrRRUgNfCxP'dfqL&NmgYC;&qlQV?V:!'DoQmC,t
GcBWr?Qw;zZQS.rTKDNtCS!!.uDzvDNF,-TXh-V!TTkaug!'YxZh'BFHtcvzzClCNz

As expected - complete garbage! This is BEFORE training.


In [8]:
# Create optimizer - this is what actually updates the model
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Training loop
print("Starting training...")
for steps in range(10000):
    # get a batch of data
    xb, yb = get_batch('train')

    # calculate loss
    logits, loss = model(xb, yb)

    # update model
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # print progress every 1000 steps
    if steps % 1000 == 0:
        print(f"Step {steps}: loss = {loss.item():.4f}")

print(f"\nFinal loss: {loss.item():.4f}")
print("Training complete!")

Starting training...
Step 0: loss = 4.5682
Step 1000: loss = 3.8270
Step 2000: loss = 3.5126
Step 3000: loss = 2.9976
Step 4000: loss = 2.8907
Step 5000: loss = 2.8970
Step 6000: loss = 2.2775
Step 7000: loss = 2.7283
Step 8000: loss = 2.5826
Step 9000: loss = 2.3953

Final loss: 2.6858
Training complete!


In [9]:
# Generate text AFTER training
context = torch.zeros((1, 1), dtype=torch.long)

# Generate 300 characters
generated = model.generate(context, max_new_tokens=300)

# Convert numbers back to text
print("Generated text AFTER training:")
print("="*40)
print(decode(generated[0].tolist()))
print("="*40)
print("\nLooks better than before right?")
print("But still not great - because this is just our SIMPLE model")
print("Next we will build the REAL Transformer model!")

Generated text AFTER training:


oulyooker ure.
I coulitholllecker?
Allil. Folerd-d y atine methafsetak-there efigesalfy wil ss.
TAUEd meve
L, d u ad lo!
GBurasthe foovisomunth

Avemos fes illsemfour br withan imysmacenon f iny
Gord.
TIbrknge, d.!s, takyowildofulldie IOLue,  pong m m s hed,&Kwo;
A.
Tynt thee f ikest;
LORonwn his
M

Looks better than before right?
But still not great - because this is just our SIMPLE model
Next we will build the REAL Transformer model!


In [10]:
# Hyperparameters - settings for our Transformer model
batch_size = 32        # how many chunks processed at once
block_size = 64        # how many characters model sees at once
max_iters = 5000       # how many training steps
eval_interval = 500    # check progress every 500 steps
learning_rate = 3e-4   # how fast model learns
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # use GPU if available
eval_iters = 200       # how many batches to estimate loss
n_embd = 64            # size of character embeddings
n_head = 4             # number of attention heads
n_layer = 4            # number of transformer blocks
dropout = 0.2          # regularization to prevent overfitting

print("Settings:")
print(f"Device: {device}")
print(f"Batch size: {batch_size}")
print(f"Block size: {block_size}")
print(f"Embedding size: {n_embd}")
print(f"Attention heads: {n_head}")
print(f"Transformer layers: {n_layer}")
print(f"Dropout: {dropout}")
print("\nAll settings ready!")

Settings:
Device: cuda
Batch size: 32
Block size: 64
Embedding size: 64
Attention heads: 4
Transformer layers: 4
Dropout: 0.2

All settings ready!


In [11]:
class Head(nn.Module):
    """ Single head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        # every character creates 3 vectors:
        self.key   = nn.Linear(n_embd, head_size, bias=False)  # what do I contain?
        self.query = nn.Linear(n_embd, head_size, bias=False)  # what am I looking for?
        self.value = nn.Linear(n_embd, head_size, bias=False)  # what do I share?
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # calculate attention scores
        wei = q @ k.transpose(-2,-1) * C**-0.5  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # weighted aggregation of values
        v = self.value(x)  # (B, T, head_size)
        out = wei @ v      # (B, T, head_size)
        return out

print("Self-Attention Head built successfully!")

Self-Attention Head built successfully!


In [12]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention running in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # create multiple heads
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # projection layer to combine all heads
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # run all heads in parallel and combine results
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

print("Multi-Head Attention built successfully!")
print(f"We have {n_head} heads working in parallel!")

Multi-Head Attention built successfully!
We have 4 heads working in parallel!


In [13]:
class FeedForward(nn.Module):
    """ Simple neural network for each character to think individually """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # expand to 4x size to think more deeply
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),  # activation function
            # compress back to original size
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

print("Feed Forward Network built successfully!")
print(f"Inner size: {4 * n_embd} (4x the embedding size)")
print("Each character now thinks independently after attention!")

Feed Forward Network built successfully!
Inner size: 256 (4x the embedding size)
Each character now thinks independently after attention!


In [14]:
class Block(nn.Module):
    """ One complete Transformer block - communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        # communication between characters
        self.sa = MultiHeadAttention(n_head, head_size)
        # individual thinking
        self.ffwd = FeedForward(n_embd)
        # layer norms to keep numbers stable
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # residual connections (x +) help gradients flow
        x = x + self.sa(self.ln1(x))   # communicate
        x = x + self.ffwd(self.ln2(x)) # think
        return x

print("Transformer Block built successfully!")
print("Each block does:")
print("  1. Characters COMMUNICATE via Multi-Head Attention")
print("  2. Characters THINK via Feed Forward Network")
print(f"We will stack {n_layer} of these blocks!")

Transformer Block built successfully!
Each block does:
  1. Characters COMMUNICATE via Multi-Head Attention
  2. Characters THINK via Feed Forward Network
We will stack 4 of these blocks!


In [15]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each character gets an embedding vector
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # each position gets an embedding vector
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # stack 4 transformer blocks
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        # final layer norm
        self.ln_f = nn.LayerNorm(n_embd)
        # convert embeddings to vocabulary predictions
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # token and position embeddings
        tok_emb = self.token_embedding_table(idx)    # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, n_embd)
        x = tok_emb + pos_emb   # (B, T, n_embd)
        # pass through transformer blocks
        x = self.blocks(x)      # (B, T, n_embd)
        x = self.ln_f(x)        # (B, T, n_embd)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # crop to block size
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Create model and move to GPU
model = GPTLanguageModel()
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print("Complete GPT Model built successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Model is running on: {device}")

Complete GPT Model built successfully!
Total parameters: 211,777
Model is running on: cuda


In [16]:
@torch.no_grad()  # don't calculate gradients here - saves memory
def estimate_loss():
    out = {}
    model.eval()  # switch to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            X, Y = X.to(device), Y.to(device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()  # switch back to training mode
    return out

# Test it!
losses = estimate_loss()
print("Loss estimation function works!")
print(f"Initial train loss: {losses['train']:.4f}")
print(f"Initial val loss:   {losses['val']:.4f}")
print("\nBoth numbers should be similar and around 4.0-4.5")

Loss estimation function works!
Initial train loss: 4.3214
Initial val loss:   4.3200

Both numbers should be similar and around 4.0-4.5


In [17]:
# Create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting Transformer training...")
print("="*50)

for iter in range(max_iters):

    # every 500 steps check progress
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {iter}: train loss = {losses['train']:.4f}  val loss = {losses['val']:.4f}")

    # get batch of data
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    # calculate loss
    logits, loss = model(xb, yb)

    # update model
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("="*50)
print("Training complete!")

Starting Transformer training...
Step 0: train loss = 4.3216  val loss = 4.3194
Step 500: train loss = 2.5047  val loss = 2.5081
Step 1000: train loss = 2.3916  val loss = 2.3985
Step 1500: train loss = 2.2959  val loss = 2.3094
Step 2000: train loss = 2.2070  val loss = 2.2273
Step 2500: train loss = 2.1355  val loss = 2.1650
Step 3000: train loss = 2.0787  val loss = 2.1140
Step 3500: train loss = 2.0222  val loss = 2.0726
Step 4000: train loss = 1.9857  val loss = 2.0352
Step 4500: train loss = 1.9531  val loss = 2.0195
Step 4999: train loss = 1.9116  val loss = 1.9797
Training complete!


In [18]:
# Generate text after training
print("Generated text AFTER Transformer training:")
print("="*50)

# start with a new line character
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# generate 500 characters
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_text)

print("="*50)
print("\nCompare this to the garbage we generated in Step 8!")
print("This is the power of the Transformer!")

Generated text AFTER Transformer training:

Roreid.

MIO:
Myshowt bes inidy my him gof the tast he are buddof a longlisce
It lords port hight cong, that wigh me denteret of gomart med my,
Aber un thee he Clower'd stare in spriceWavorvey.

Fir

DULANES:
Beow! Joldny he mutarlionce I dow woul my prothy,
Then thou: of you to deacen daild
I wall coutoston titme thisembey,
Head mede oft in preate.

RUCINIULI:
I it abll, the cowards?

MERWANENTRENR:
I ford, mouneved! Xoyet Crie swareac cose,
To lord swer, a the lakigh Mall, beraact you,
Than tr

Compare this to the garbage we generated in Step 8!
This is the power of the Transformer!


In [19]:
# Save the model
torch.save(model.state_dict(), 'gpt_model.pth')
print("Model saved successfully as 'gpt_model.pth'!")

# Save the vocabulary too
import json
vocab_data = {
    'chars': chars,
    'vocab_size': vocab_size,
    'stoi': stoi,
    'itos': {str(k): v for k, v in itos.items()}
}
with open('vocab.json', 'w') as f:
    json.dump(vocab_data, f)
print("Vocabulary saved as 'vocab.json'!")

# Verify files are saved
import os
print(f"\nModel size: {os.path.getsize('gpt_model.pth') / 1024:.1f} KB")
print(f"Vocab size: {os.path.getsize('vocab.json') / 1024:.1f} KB")
print("\nBoth files saved! Download them from the Colab file browser on the left!")

Model saved successfully as 'gpt_model.pth'!
Vocabulary saved as 'vocab.json'!

Model size: 1123.8 KB
Vocab size: 1.6 KB

Both files saved! Download them from the Colab file browser on the left!


In [20]:
def generate_shakespeare(prompt=None, num_chars=500):
    """
    Generate Shakespeare-like text using our trained GPT model

    Args:
        prompt: starting text (optional)
        num_chars: how many characters to generate
    """
    model.eval()

    if prompt is None:
        # start from scratch
        context = torch.zeros((1, 1), dtype=torch.long, device=device)
    else:
        # start from given prompt
        encoded = encode(prompt)
        context = torch.tensor(encoded, dtype=torch.long, device=device).unsqueeze(0)

    # generate text
    generated = model.generate(context, max_new_tokens=num_chars)
    generated_text = decode(generated[0].tolist())

    return generated_text

# Test 1 - generate from scratch
print("TEST 1 - Generate from scratch:")
print("="*50)
print(generate_shakespeare(num_chars=300))

print("\n")

# Test 2 - generate from a prompt
print("TEST 2 - Generate from prompt 'KING:':")
print("="*50)
print(generate_shakespeare(prompt="KING:", num_chars=300))

TEST 1 - Generate from scratch:

HED HO:
A you bidime, dont har love at.

ROMERUS:
So'le in man, you neady I thebly blieds,
With tase more, this.

ROUCEO:
Agion:
Heest promend in have hom ast thour puctnrim,
I ? eycourse:
All conge! my monese! I hist prit-hast not up,
The But thime taked weare!

KANG EDWIO:
Nood is as of the thime?


TEST 2 - Generate from prompt 'KING:':
KING:

Gord God the nuer, not ton, and theal, she your mink the by reseaks.
Butch'd I nefull deminted for dintmone elly
Well rok, in this mos have is to me kine ood my moty:
Buton two alood of if will of sexeparttrifing;
Ten Lord thee Youll our make grese
To ma!

GOKENO:
A hat are spupinciupher to kince.
